# On-Chain Pool Data — FinTech 590
Queries live Uniswap V3 pool state directly from each chain using the saved ABIs.

**Run order:** `defi_pipeline.ipynb` → `historical_data.ipynb` → `database.ipynb` → this notebook.

**What this fetches per pool:**
| Field | Source | Description |
|-------|--------|-------------|
| `sqrt_price_x96` | `slot0()` | Encoded current price (raw uint160) |
| `tick` | `slot0()` | Current tick index |
| `liquidity` | `liquidity()` | Active in-range liquidity (raw uint128) |
| `fee` | `fee()` | Fee tier in pips (confirms on-chain value) |

Uses public RPC endpoints by default — no API key needed.  
Add `RPC_ETHEREUM`, `RPC_ARBITRUM`, `RPC_OPTIMISM`, `RPC_BASE`, `RPC_POLYGON` to `.env` to use private endpoints.

## 0. Install Dependencies

In [1]:
import subprocess, sys

def ensure(*packages):
    for pkg in packages:
        try:
            __import__(pkg.replace("-", "_"))
        except ImportError:
            print(f"[setup] Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

ensure("pandas", "python-dotenv", "web3", "pyarrow")
print("All dependencies ready.")

[setup] Installing python-dotenv...
All dependencies ready.


## 1. Config

In [2]:
import os, json, time, pathlib
from datetime import date
import pandas as pd
from dotenv import load_dotenv
from web3 import Web3

load_dotenv()

# Public RPC endpoints — free, no key required
# Override any of these in .env for better reliability (Alchemy/Infura)
RPC_URLS = {
    "Ethereum": os.getenv("RPC_ETHEREUM", "https://eth.llamarpc.com"),
    "Arbitrum": os.getenv("RPC_ARBITRUM", "https://arb1.arbitrum.io/rpc"),
    "Optimism": os.getenv("RPC_OPTIMISM", "https://mainnet.optimism.io"),
    "Base":     os.getenv("RPC_BASE",     "https://mainnet.base.org"),
    "Polygon":  os.getenv("RPC_POLYGON",  "https://rpc.ankr.com/polygon"),  # ankr: no auth required
}

POOLS_PARQUET   = pathlib.Path("data/top_pools.parquet")
ABI_DIR         = pathlib.Path("pool_abis")
ONCHAIN_PARQUET = pathlib.Path("data/pool_onchain.parquet")
DELAY           = 0.25   # seconds between RPC calls
MAX_RETRIES     = 3      # retries on transient RPC errors

# Minimal ERC-20 ABI for fetching token decimals
ERC20_ABI = [{"inputs": [], "name": "decimals", "outputs": [{"type": "uint8"}], "stateMutability": "view", "type": "function"}]

_decimals_cache: dict = {}

def get_decimals(w3: Web3, token_address: str) -> int:
    """Fetch and cache ERC-20 decimals for a token."""
    key = token_address.lower()
    if key not in _decimals_cache:
        try:
            contract = w3.eth.contract(address=Web3.to_checksum_address(token_address), abi=ERC20_ABI)
            _decimals_cache[key] = contract.functions.decimals().call()
        except Exception:
            _decimals_cache[key] = 18  # fallback
    return _decimals_cache[key]

def call_with_retry(fn, retries: int = MAX_RETRIES, backoff: float = 1.5):
    """Call a Web3 function with exponential backoff on transient errors."""
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            msg = str(e).lower()
            if any(k in msg for k in ("429", "rate", "timeout", "connection", "header not found")):
                if attempt < retries - 1:
                    time.sleep(backoff ** attempt)
                    continue
            raise
    raise RuntimeError(f"Failed after {retries} retries")

print("RPC endpoints:")
for chain, url in RPC_URLS.items():
    src = "(custom)" if os.getenv(f"RPC_{chain.upper()}") else "(public)"
    print(f"  {chain:<10} {src}  {url}")

RPC endpoints:
  Ethereum   (public)  https://eth.llamarpc.com
  Arbitrum   (public)  https://arb1.arbitrum.io/rpc
  Optimism   (public)  https://mainnet.optimism.io
  Base       (public)  https://mainnet.base.org
  Polygon    (public)  https://rpc.ankr.com/polygon


## 2. Load Pool List & Check Connectivity

In [3]:
pools = pd.read_parquet(POOLS_PARQUET)

# Keep only pools that have a saved ABI
pools = pools[pools["address"].apply(lambda a: (ABI_DIR / f"{a}.json").exists())].copy()
print(f"Pools with ABI: {len(pools)} / {len(pd.read_parquet(POOLS_PARQUET))}")
print()

# Test RPC connectivity per chain
web3_clients = {}
for chain in pools["chain"].unique():
    if chain not in RPC_URLS:
        print(f"  [{chain}] no RPC URL configured — skipping")
        continue
    try:
        w3 = Web3(Web3.HTTPProvider(RPC_URLS[chain]))
        block = w3.eth.block_number   # raises if unreachable
        web3_clients[chain] = w3
        print(f"  [{chain}] connected  (latest block: {block:,})")
    except Exception as e:
        print(f"  [{chain}] FAILED to connect ({e}) — pools on this chain will be skipped")

# Filter to chains with working RPC
pools = pools[pools["chain"].isin(web3_clients)].copy()
print(f"\nReady to query {len(pools)} pools across {len(web3_clients)} chains.")

Pools with ABI: 47 / 47

  [Ethereum] connected  (latest block: 24,751,096)
  [Arbitrum] connected  (latest block: 446,299,598)
  [Optimism] connected  (latest block: 149,521,139)
  [Base] connected  (latest block: 43,925,854)
  [Polygon] FAILED to connect ({'code': -32000, 'message': 'Unauthorized: You must authenticate your request with an API key. Create an account on https://www.ankr.com/rpc/ and generate your personal API key for free.'}) — pools on this chain will be skipped

Ready to query 44 pools across 4 chains.


## 3. Query On-Chain State

Calls three view functions on each pool contract:
- `slot0()` → current price (sqrtPriceX96) and tick
- `liquidity()` → active in-range liquidity
- `fee()` → fee tier in pips

In [6]:
records = []
fetched_at = str(date.today())

for chain, group in pools.groupby("chain"):
    w3 = web3_clients[chain]
    print(f"\n[{chain}]")

    for _, pool in group.iterrows():
        addr  = pool["address"]
        label = f"{pool['token0']}/{pool['token1']} {pool['fee_tier']/1e4:.2f}%"
        print(f"  {addr[:10]}...  {label}", end="", flush=True)

        try:
            abi      = json.loads((ABI_DIR / f"{addr}.json").read_text())
            contract = w3.eth.contract(address=Web3.to_checksum_address(addr), abi=abi)

            slot0     = call_with_retry(contract.functions.slot0().call)
            liquidity = call_with_retry(contract.functions.liquidity().call)
            fee       = call_with_retry(contract.functions.fee().call)

            sqrt_price_x96 = slot0[0]
            tick           = slot0[1]

            # Fetch token addresses from pool contract for decimal lookup
            token0_addr = call_with_retry(contract.functions.token0().call)
            token1_addr = call_with_retry(contract.functions.token1().call)
            dec0 = get_decimals(w3, token0_addr)
            dec1 = get_decimals(w3, token1_addr)

            # Human-readable price: (sqrtPriceX96 / 2^96)^2 adjusted for decimals
            # Gives price of token0 denominated in token1
            raw_price = (sqrt_price_x96 / (2 ** 96)) ** 2
            price = raw_price * (10 ** dec0) / (10 ** dec1)

            records.append({
                "address":        addr,
                "chain":          chain,
                "token0":         pool["token0"],
                "token1":         pool["token1"],
                "sqrt_price_x96": str(sqrt_price_x96),
                "tick":           tick,
                "liquidity":      str(liquidity),
                "fee":            fee,
                "dec0":           dec0,
                "dec1":           dec1,
                "price":          price,
                "fetched_at":     fetched_at,
            })
            print(f"  ✓  tick={tick:,}  liquidity={liquidity:.2e}  price={price:.6f} {pool['token1']}/{pool['token0']}")

        except Exception as e:
            print(f"  ✗  {e}")

        time.sleep(DELAY)

print(f"\nQueried {len(records)} pools successfully.")


[Arbitrum]
  0x2f5e87C9...  WBTC/WETH 0.05%  ✓  tick=265,263  liquidity=4.65e+17  price=33.086768 WETH/WBTC
  0x0E483131...  WBTC/USDC 0.05%  ✓  tick=64,912  liquidity=1.43e+12  price=65915.245738 USDC/WBTC
  0xd13040d4...  RAIN/WETH 0.01%  ✓  tick=-123,664  liquidity=4.56e+22  price=0.000004 WETH/RAIN
  0xbE3aD6a5...  USDC/USDT 0.01%  ✓  tick=5  liquidity=1.16e+16  price=1.000524 USDT/USDC
  0x80A9ae39...  WETH/GMX 1.00%  ✓  tick=57,228  liquidity=2.41e+20  price=305.695927 GMX/WETH
  0x689C96ce...  WBTC/ARB 0.30%  ✓  tick=365,286  liquidity=1.03e+18  price=730113.313529 ARB/WBTC
  0x9B42809a...  WBTC/CBBTC 0.01%  ✓  tick=-14  liquidity=9.90e+11  price=0.998671 CBBTC/WBTC
  0x1DFc1054...  HEGIC/WETH 0.05%  ✓  tick=-118,752  liquidity=2.69e+23  price=0.000007 WETH/HEGIC
  0x5886e46E...  USDC/THBILL 0.01%  ✓  tick=-174  liquidity=4.71e+13  price=0.982778 THBILL/USDC
  0xD8D4314f...  LON/USDC 0.30%  ✓  tick=-289,584  liquidity=1.31e+18  price=0.265573 USDC/LON
  0xdf04FA6e...  CAPX/USDC

## 4. Save to Parquet

In [7]:
if not records:
    print("No records collected — check RPC connectivity and ABI availability.")
else:
    df = pd.DataFrame(records)
    df.to_parquet(ONCHAIN_PARQUET, index=False, engine="pyarrow")
    print(f"Saved {len(df)} rows to {ONCHAIN_PARQUET}")
    print()
    print(df[["chain", "token0", "token1", "fee", "tick", "price", "dec0", "dec1", "fetched_at"]])

Saved 44 rows to data\pool_onchain.parquet

       chain    token0    token1    fee    tick         price  dec0  dec1  \
0   Arbitrum      WBTC      WETH    500  265263  3.308677e+01     8    18   
1   Arbitrum      WBTC      USDC    500   64912  6.591525e+04     8     6   
2   Arbitrum      RAIN      WETH    100 -123664  4.262334e-06    18    18   
3   Arbitrum      USDC      USDT    100       5  1.000524e+00     6     6   
4   Arbitrum      WETH       GMX  10000   57228  3.056959e+02    18    18   
5   Arbitrum      WBTC       ARB   3000  365286  7.301133e+05     8    18   
6   Arbitrum      WBTC     CBBTC    100     -14  9.986708e-01     8     8   
7   Arbitrum     HEGIC      WETH    500 -118752  6.965383e-06    18    18   
8   Arbitrum      USDC    THBILL    100    -174  9.827778e-01     6     6   
9   Arbitrum       LON      USDC   3000 -289584  2.655727e-01    18     6   
10  Arbitrum      CAPX      USDC    100 -294390  1.642301e-01    18     6   
11      Base      WETH     CBBTC

## 5. Sanity Checks

In [8]:
if not records:
    print("No data to check — Cell 4 produced no records.")
else:
    # Verify fee tier matches DeFiLlama data
    pools_ref = pd.read_parquet(POOLS_PARQUET)
    merged = df.merge(pools_ref[["address", "chain", "fee_tier"]], on=["address", "chain"])
    mismatches = merged[merged["fee"] != merged["fee_tier"]]

    if mismatches.empty:
        print("✓ All on-chain fee tiers match DeFiLlama data")
    else:
        print(f"✗ {len(mismatches)} fee tier mismatches:")
        print(mismatches[["chain", "token0", "token1", "fee", "fee_tier"]])

    print()
    print("Pools per chain:")
    print(df.groupby("chain")[["address"]].count().rename(columns={"address": "pools"}).to_string())

✓ All on-chain fee tiers match DeFiLlama data

Pools per chain:
          pools
chain          
Arbitrum     11
Base         12
Ethereum     20
Optimism      1
